In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()

        self.c1 = nn.Conv2d(1, 6, kernel_size=5, stride=1)

        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)

        self.c3 = nn.Conv2d(6, 16, kernel_size=5, stride=1)

        self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)

        self.c5 = nn.Conv2d(16, 120, kernel_size=5, stride=1)

        self.f6 = nn.Linear(120, 84)

        self.out = nn.Linear(84, 10)

        self.act = nn.Tanh()

    def forward(self, x):
        x = self.act(self.c1(x))
        x = self.s2(x)

        x = self.act(self.c3(x))
        x = self.s4(x)

        x = self.act(self.c5(x))

        x = x.view(x.size(0), -1)

        x = self.act(self.f6(x))
        x = self.out(x)

        return x

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transform = transforms.Compose([
        transforms.Pad(2),
        transforms.ToTensor()
    ])

    train_dataset = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    model = LeNet5().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    epochs = 10

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss / len(train_loader):.4f}")

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
if __name__ == "__main__":
    main()

Epoch [1/10] Loss: 0.2914
Epoch [2/10] Loss: 0.0953
Epoch [3/10] Loss: 0.0646
Epoch [4/10] Loss: 0.0489
Epoch [5/10] Loss: 0.0376
Epoch [6/10] Loss: 0.0302
Epoch [7/10] Loss: 0.0267
Epoch [8/10] Loss: 0.0198
Epoch [9/10] Loss: 0.0196
Epoch [10/10] Loss: 0.0154
Test Accuracy: 98.57%
